# Solutions · 5. Robust regression: the Huber loss and quantile regression

Worked solutions to the three exercises of [Lecture 5](../../notebooks/part2_shape/05_outlier_robust_regression_huber_irls_and_quantile_loss.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: Huber Loss vs. Quadratic L2 Loss & IRLS Weighting")
rng = np.random.default_rng(42)
N = 45
T_norm = np.linspace(-2.0, 2.0, N)
X_des = np.column_stack([np.ones(N), T_norm])
w_true = np.array([12.0, 3.5])
y_clean = X_des @ w_true + rng.normal(0.0, 0.25, size=N)

# Corrupt 3 sensor readings with spikes of +25.0 (~100 sigma):
y_spiked = y_clean.copy()
spike_idx = [5, 22, 38]
y_spiked[spike_idx] += 25.0

w_ols = linear_physics.ridge_regression(X_des, y_spiked, lam=0.0)
huber_res = linear_physics.huber_irls_regression(X_des, y_spiked, delta=1.0)
w_huber = huber_res["w"]

print("True parameters   [intercept, slope]:", w_true)
print("Corrupted OLS     [intercept, slope]:", np.round(w_ols, 4), f"(L2 error = {np.linalg.norm(w_ols - w_true):.4f})")
print("Robust Huber IRLS [intercept, slope]:", np.round(w_huber, 4), f"(L2 error = {np.linalg.norm(w_huber - w_true):.4f})")
assert np.linalg.norm(w_huber - w_true) < 0.15 * np.linalg.norm(w_ols - w_true)
heading("Demonstration 2: Inspecting Converged IRLS Sample Weights on Inliers vs. Spikes")
weights_conv = huber_res["sample_weights"]
res_conv = huber_res["residuals"]
inlier_mask = np.ones(N, dtype=bool)
inlier_mask[spike_idx] = False

print(f"Mean IRLS weight across 42 normal inliers: {np.mean(weights_conv[inlier_mask]):.4f}")
for idx in spike_idx:
    print(f"Spike sample #{idx:2d} | Residual r_i = {res_conv[idx]:+7.3f} | Converged IRLS weight w_i = {weights_conv[idx]:.4f}")
assert np.all(weights_conv[spike_idx] < 0.08) and np.mean(weights_conv[inlier_mask]) > 0.98
heading("Demonstration 3: Pinball Quantile Envelopes (10th, 50th Median, 90th Percentiles)")
w_q10 = linear_physics.quantile_irls_regression(X_des, y_spiked, tau=0.10)
w_q50 = linear_physics.quantile_irls_regression(X_des, y_spiked, tau=0.50)
w_q90 = linear_physics.quantile_irls_regression(X_des, y_spiked, tau=0.90)

y_q10, y_q50, y_q90 = X_des @ w_q10, X_des @ w_q50, X_des @ w_q90
print("10th Percentile weights (tau=0.10):", np.round(w_q10, 4))
print("50th Median L1 weights  (tau=0.50):", np.round(w_q50, 4), f"(L2 error vs true = {np.linalg.norm(w_q50 - w_true):.4f})")
print("90th Percentile weights (tau=0.90):", np.round(w_q90, 4))
assert np.all(y_q10 < y_q50) and np.all(y_q50 < y_q90)
assert np.linalg.norm(w_q50 - w_true) < 0.20 * np.linalg.norm(w_ols - w_true)
heading("Demonstration 4: Huber Threshold delta Sweep (From L1 Limit to OLS Limit)")
print(f"{'Huber delta':>12s} | {'Intercept w_0':>13s} | {'Slope w_1':>10s} | {'Param L2 Error':>14s} | {'Spike Weight':>12s}")
print("-" * 70)
for d_val in [0.1, 0.3, 1.0, 5.0, 50.0]:
    res_d = linear_physics.huber_irls_regression(X_des, y_spiked, delta=d_val)
    wh = res_d["w"]
    print(f"{d_val:12.1f} | {wh[0]:13.4f} | {wh[1]:10.4f} | {np.linalg.norm(wh - w_true):14.4f} | {res_d['sample_weights'][spike_idx[0]]:12.4f}")
heading("Inside the Algorithm: By-Hand Huber IRLS Loop vs. the Library")
w_hand = np.linalg.lstsq(X_des, y_spiked, rcond=None)[0]
for _ in range(100):
    w_prev = w_hand.copy()
    r_abs = np.abs(y_spiked - X_des @ w_hand)
    sw = np.where(r_abs <= 1.0, 1.0, 1.0 / np.maximum(r_abs, 1e-15))
    W_half = np.sqrt(sw)[:, None]
    w_hand = np.linalg.lstsq(X_des * W_half, y_spiked * W_half.ravel(), rcond=None)[0]
    if np.max(np.abs(w_hand - w_prev)) < 1e-9:
        break
diff_huber = float(np.max(np.abs(w_hand - w_huber)))
print("by hand (pure NumPy):", np.round(w_hand, 6))
print("library (engml_2_applied)     :", np.round(w_huber, 6))
print(f"Max discrepancy     : {diff_huber:.2e}")
assert diff_huber < 1e-14
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Verify on `y_spiked` that the empirical fraction of observations falling below the fitted $\tau = 0.50$ median regression line (`w_q50`) lies within $0.50 \pm 0.08$, and that the $\tau = 0.50$ median slope is within $0.15$ of the true slope $3.5$ despite the three $+25.0$ gross outliers.

In [2]:
heading("Exercise 1: Quantile Split Balance & Median Slope Verification")
frac_below_50 = float(np.mean(y_spiked <= (X_des @ w_q50)))
slope_err_50 = abs(float(w_q50[1]) - float(w_true[1]))
print(f"Empirical fraction <= median line = {frac_below_50:.3f} | Median slope error = {slope_err_50:.4f}")
assert abs(frac_below_50 - 0.50) <= 0.08 and slope_err_50 < 0.15


Exercise 1: Quantile Split Balance & Median Slope Verification
Empirical fraction <= median line = 0.511 | Median slope error = 0.0077


**Discussion.** By first-order optimality of the Pinball check loss $\rho_\tau(r)$, an exact $\tau$-quantile regression line partitions the dataset so that approximately $100\tau\%$ of the observations lie at or below the fitted line.

---

## Exercise 2

**Problem.** Demonstrate the **high-leverage X-outlier vulnerability** of standard M-estimators: corrupt both the input $x_{44} = 2.0 \to 25.0$ and target $y_{44} \to -100.0$ (a simultaneous input-output leverage point) and compare `huber_irls_regression` with and without a $3\sigma_x$ input leverage pre-filter.

In [3]:
heading("Exercise 2: High-Leverage X-Outlier Breakdown Audit & Leverage Guardrail")
X_lev = X_des.copy()
y_lev = y_spiked.copy()
X_lev[-2:, 1] = 250.0
y_lev[-2:] = -200.0
w_huber_unfiltered = linear_physics.huber_irls_regression(X_lev, y_lev, delta=1.0)["w"]

# Guardrail: Median + MAD input domain screen |x - med(x)| <= 3 * 1.4826 * MAD(x):
x_col = X_lev[:, 1]
mad_x = 1.4826 * float(np.median(np.abs(x_col - np.median(x_col))))
keep = np.abs(x_col - np.median(x_col)) <= 3.0 * mad_x
w_huber_guarded = linear_physics.huber_irls_regression(X_lev[keep], y_lev[keep], delta=1.0)["w"]
print("Unfiltered Huber on X-leverage outlier:", np.round(w_huber_unfiltered, 3))
print("MAD-Guarded Huber on X-leverage data  :", np.round(w_huber_guarded, 3), "(True = [12.0, 3.5])")
assert np.linalg.norm(w_huber_guarded - w_true) < 0.25 and np.linalg.norm(w_huber_unfiltered - w_true) > 1.5


Exercise 2: High-Leverage X-Outlier Breakdown Audit & Leverage Guardrail
Unfiltered Huber on X-leverage outlier: [12.116 -0.848]
MAD-Guarded Huber on X-leverage data  : [12.096  3.524] (True = [12.0, 3.5])


**Discussion.** While Huber and $L_1$ loss bound the influence of vertical $y$-outliers, their influence function $\psi_\delta(r)\mathbf{x}_i$ is still proportional to $\mathbf{x}_i$. Pairing Huber IRLS with an input domain guardrail (MAD or Mahalanobis OOD screen) protects against both $x$- and $y$-outliers.

---

## Exercise 3

**Problem.** Implement a function `auto_mad_huber_regression(X: np.ndarray, y: np.ndarray, c: float = 1.345) -> dict` that computes an initial $\tau=0.50$ median regression fit, estimates the robust noise scale $\hat{\sigma}_{\mathrm{MAD}} = 1.4826\,\mathrm{median}(|\mathbf{r}_0 - \mathrm{median}(\mathbf{r}_0)|)$, and runs `huber_irls_regression` with the self-tuning threshold $\delta = c\,\hat{\sigma}_{\mathrm{MAD}}$. Verify it on `y_spiked`.

In [4]:
heading("Exercise 3: Implement Self-Tuning MAD-Scaled Huber Regression")
def auto_mad_huber_regression(X: np.ndarray, y: np.ndarray, c: float = 1.345) -> dict:
    """Self-tuning Huber regression using median-regression MAD scale estimation."""
    w_init = linear_physics.quantile_irls_regression(X, y, tau=0.50)
    r0 = y - X @ w_init
    sigma_mad = 1.4826 * float(np.median(np.abs(r0 - np.median(r0))))
    delta_auto = max(1e-6, c * sigma_mad)
    res = linear_physics.huber_irls_regression(X, y, delta=delta_auto)
    res["sigma_mad"] = sigma_mad
    res["delta_auto"] = delta_auto
    return res

auto_h = auto_mad_huber_regression(X_des, y_spiked)
print(f"Estimated robust scale sigma_MAD = {auto_h['sigma_mad']:.4f} (True sigma = 0.2500)")
print(f"Self-tuned Huber delta           = {auto_h['delta_auto']:.4f} | Fitted w = {np.round(auto_h['w'], 4)}")
assert abs(auto_h["sigma_mad"] - 0.25) < 0.08 and np.linalg.norm(auto_h["w"] - w_true) < 0.15


Exercise 3: Implement Self-Tuning MAD-Scaled Huber Regression
Estimated robust scale sigma_MAD = 0.2242 (True sigma = 0.2500)
Self-tuned Huber delta           = 0.3015 | Fitted w = [12.0425  3.5097]


**Discussion.** Scaling Huber's threshold via the median absolute deviation ($\delta = 1.345 \times 1.4826\,\mathrm{MAD}$) makes the fit independent of the units of the response and leaves no threshold to choose by hand. Here the estimated scale is $0.224$ (true value $0.25$).